# CircuitSight — YOLOv8 Defect Detector
### Full PCB image in, bounding boxes + defect class out (Stage 1)

Dataset: `akhatova/pcb-defects` (PCB_DATASET), 6 defect classes
Detector: **YOLOv8** (COCO pretrained, fine-tuned on the full-board images + VOC boxes)

The EfficientNet-B0 classifier only works on defect crops. Given a whole board it has no way to say "no defect" and returns an overconfident wrong class. This notebook trains the missing piece: a detector that finds *where* the defects are. Flow at inference: **full board -> YOLO boxes -> (optional) EfficientNet on each crop**.

**Same board-level split as the classifier notebooks.** Photos come from only ~10 base boards (`01_spur_02.jpg` is board `01`). A random per-image split would put photos of the same board in train and test and inflate mAP, so whole boards are assigned to train / val / test using the split already stored in `data/dataset_v2/splits.csv`. Classifier and detector are therefore evaluated on the same unseen boards.

**Pipeline:** VOC XML -> YOLO txt (board-level split) -> Train -> Evaluate (mAP) -> Visualise -> Save `.pt`

## 1. Setup & Imports

In [ ]:
# !pip install ultralytics   # run once if needed
import os, time, json, shutil, random
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2, yaml, torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2. Config & Directory Paths

```
backend/
├── dataset/PCB_DATASET/{images,Annotations}/<Class>/   <- full boards + VOC boxes
├── data/
│   ├── dataset_v2/splits.csv                           <- board-level split (from build_dataset.py)
│   └── yolo_dataset/                                   <- generated YOLO-format copy (cache)
├── models/yolov8_pcb_best.pt                           <- saved detector
├── notebooks/yolov8_pcb_detection.ipynb
└── results/yolov8/                                     <- curves, PR curves, confusion matrix, metrics
```

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent.resolve() if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR.resolve()

DATASET_ROOT = PROJECT_ROOT / "dataset" / "PCB_DATASET"
IMAGES_DIR   = DATASET_ROOT / "images"
ANNOT_DIR    = DATASET_ROOT / "Annotations"
SPLITS_CSV   = PROJECT_ROOT / "data" / "dataset_v2" / "splits.csv"
REPORT_JSON  = PROJECT_ROOT / "data" / "dataset_v2" / "dataset_report.json"

YOLO_DIR    = PROJECT_ROOT / "data" / "yolo_dataset"
MODEL_DIR   = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results" / "yolov8"
for d in [MODEL_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)

assert IMAGES_DIR.exists() and ANNOT_DIR.exists(), "Dataset folders not found"
assert SPLITS_CSV.exists(), "splits.csv not found - run `python build_dataset.py` from the project root first."

# ---- training config ----
MODEL_NAME = "yolov8s.pt"   # yolov8n.pt = lighter/faster, yolov8m.pt = more accurate
IMG_SIZE   = 1024           # boards are ~3000 px wide and defects are tiny; 1280 is better if the GPU allows
BATCH_SIZE = 8              # drop to 4 on CUDA out-of-memory
NUM_EPOCHS = 100
PATIENCE   = 20             # early stopping on val mAP
NUM_WORKERS = 0 if os.name == "nt" else 2   # 0 on Windows avoids DataLoader issues in Jupyter
REBUILD_YOLO = True         # set False to reuse the cached YOLO folder

# Classes come from the images/ folder names (lower-cased), same as the classifier notebooks
CLASSES = sorted(d.name.lower() for d in IMAGES_DIR.iterdir() if d.is_dir())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
print(f"{len(CLASSES)} classes:", CLASSES)

## 3. Board-level split (reuse `splits.csv`)

`splits.csv` is crop-level, so it is collapsed to a `board -> split` map. Every full-board photo is then assigned by its board id (the number at the start of the file name).

In [ ]:
manifest = pd.read_csv(SPLITS_CSV, dtype={"board": str})
SPLIT_ID = json.load(open(REPORT_JSON))["split_id"]

# each board must belong to exactly one split
per_board = manifest.groupby("board")["split"].nunique()
assert (per_board == 1).all(), f"Boards appear in more than one split: {per_board[per_board > 1].index.tolist()}"
board_split = manifest.groupby("board")["split"].first().to_dict()
print("split_id:", SPLIT_ID)
for s in ["train", "val", "test"]:
    print(f"  {s:5s} boards:", sorted(b for b, v in board_split.items() if v == s))

def board_of(path):
    return Path(path).stem.split("_")[0]          # '01_spur_02' -> '01'

records = []   # (img_path, xml_path, split)
missing = Counter()
for folder in sorted(IMAGES_DIR.iterdir()):
    if not folder.is_dir(): continue
    for img in sorted(folder.glob("*")):
        if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}: continue
        xml = ANNOT_DIR / folder.name / (img.stem + ".xml")
        b = board_of(img)
        if not xml.exists():      missing["no_xml"] += 1; continue
        if b not in board_split:  missing["board_not_in_splits"] += 1; continue
        records.append((img, xml, board_split[b]))
print("\nUsable board photos:", len(records), "| skipped:", dict(missing))
print(Counter(r[2] for r in records))

## 4. Convert VOC XML -> YOLO format

YOLO wants one `.txt` per image with `class cx cy w h` (normalised 0-1). Boxes are clipped to the image and degenerate boxes are dropped. Image files are prefixed with their class folder to avoid name clashes.

In [ ]:
def voc_to_yolo(xml_path, folder_cls):
    root = ET.parse(xml_path).getroot()
    sz = root.find("size"); W, H = int(sz.find("width").text), int(sz.find("height").text)
    lines, skipped = [], 0
    for obj in root.findall("object"):
        name = (obj.find("name").text or folder_cls).strip().lower().replace(" ", "_")
        if name not in class_to_idx: skipped += 1; continue
        bb = obj.find("bndbox")
        x1, y1, x2, y2 = [float(bb.find(k).text) for k in ("xmin", "ymin", "xmax", "ymax")]
        x1, x2 = max(0, min(x1, x2)), min(W, max(x1, x2))
        y1, y2 = max(0, min(y1, y2)), min(H, max(y1, y2))
        if x2 - x1 < 2 or y2 - y1 < 2: skipped += 1; continue
        lines.append(f"{class_to_idx[name]} {(x1+x2)/2/W:.6f} {(y1+y2)/2/H:.6f} {(x2-x1)/W:.6f} {(y2-y1)/H:.6f}")
    return lines, skipped

if REBUILD_YOLO and YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)
skipped_total = 0
if not (YOLO_DIR / "data.yaml").exists():
    for img, xml, split in records:
        (YOLO_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (YOLO_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        lines, sk = voc_to_yolo(xml, img.parent.name.lower()); skipped_total += sk
        stem = f"{img.parent.name.lower()}__{img.stem}"
        shutil.copy2(img, YOLO_DIR / "images" / split / (stem + img.suffix.lower()))
        (YOLO_DIR / "labels" / split / (stem + ".txt")).write_text("\n".join(lines))
    cfg = {"path": YOLO_DIR.resolve().as_posix(), "train": "images/train", "val": "images/val",
           "test": "images/test", "names": {i: c for i, c in enumerate(CLASSES)}}
    yaml.dump(cfg, open(YOLO_DIR / "data.yaml", "w"), sort_keys=False)
print("Skipped boxes:", skipped_total)
print(open(YOLO_DIR / "data.yaml").read())

## 5. Sanity checks: box counts, box sizes, one drawn sample

In [ ]:
rows = []
for split in ["train", "val", "test"]:
    for lf in (YOLO_DIR / "labels" / split).glob("*.txt"):
        for ln in lf.read_text().splitlines():
            c, cx, cy, w, h = ln.split(); rows.append((split, CLASSES[int(c)], float(w), float(h)))
box_df = pd.DataFrame(rows, columns=["split", "cls", "w", "h"])
counts = box_df.groupby(["cls", "split"]).size().unstack(fill_value=0)
print("Boxes per class / split:"); print(counts)
print("\nMedian box size (% of image):"); print((box_df.groupby("cls")[["w", "h"]].median() * 100).round(2))

counts.plot(kind="bar", figsize=(8, 4), title="Boxes per class and split")
plt.ylabel("Boxes"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "box_distribution.png", dpi=150); plt.show()

# draw one training label to confirm boxes line up with the board
lf = random.choice(list((YOLO_DIR / "labels/train").glob("*.txt")))
im = cv2.cvtColor(cv2.imread(str(next((YOLO_DIR / "images/train").glob(lf.stem + ".*")))), cv2.COLOR_BGR2RGB)
H, W = im.shape[:2]
for ln in lf.read_text().splitlines():
    c, cx, cy, w, h = ln.split(); cx, cy, w, h = float(cx)*W, float(cy)*H, float(w)*W, float(h)*H
    p1, p2 = (int(cx - w/2), int(cy - h/2)), (int(cx + w/2), int(cy + h/2))
    cv2.rectangle(im, p1, p2, (255, 0, 0), 4)
    cv2.putText(im, CLASSES[int(c)], (p1[0], max(30, p1[1] - 8)), cv2.FONT_HERSHEY_SIMPLEX, 2, (255, 0, 0), 4)
plt.figure(figsize=(12, 6)); plt.imshow(im); plt.axis("off"); plt.title(lf.stem)
plt.savefig(RESULTS_DIR / "sample_labels.png", dpi=150); plt.show()

## 6. Model & training

YOLOv8 starts from COCO weights and is fine-tuned end to end. Augmentation: flips are on (PCB defects have no "up"), mosaic is on, colour jitter is light, and arbitrary-angle rotation is off. Training stops early if val mAP does not improve for `PATIENCE` epochs. Ultralytics writes curves and the confusion matrix to `results/yolov8/train/`.

In [ ]:
model = YOLO(MODEL_NAME)
start_time = time.time()
train_res = model.train(
    data=str(YOLO_DIR / "data.yaml"),
    epochs=NUM_EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE, device=device,
    patience=PATIENCE, seed=SEED, workers=NUM_WORKERS,
    project=str(RESULTS_DIR), name="train", exist_ok=True,
    flipud=0.5, fliplr=0.5, degrees=0, mosaic=1.0, close_mosaic=10,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, scale=0.3, translate=0.1,
    plots=True,
)
total_time = time.time() - start_time
best_path = RESULTS_DIR / "train" / "weights" / "best.pt"
print(f"\nTraining complete in {total_time/60:.1f} min. Best weights: {best_path}")

## 7. Training curves

In [ ]:
res_csv = pd.read_csv(RESULTS_DIR / "train" / "results.csv"); res_csv.columns = [c.strip() for c in res_csv.columns]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(res_csv["train/box_loss"], label="train box"); axes[0].plot(res_csv["val/box_loss"], label="val box")
axes[0].plot(res_csv["train/cls_loss"], label="train cls"); axes[0].plot(res_csv["val/cls_loss"], label="val cls")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(res_csv["metrics/mAP50(B)"], label="mAP50"); axes[1].plot(res_csv["metrics/mAP50-95(B)"], label="mAP50-95")
axes[1].set_title("Validation mAP"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR / "training_curves.png", dpi=150); plt.show()

## 8. Evaluation on the held-out test boards

In [ ]:
best = YOLO(str(best_path))
m = best.val(data=str(YOLO_DIR / "data.yaml"), split="test", imgsz=IMG_SIZE, device=device,
             project=str(RESULTS_DIR), name="test_eval", exist_ok=True, plots=True)

print(f"mAP50: {m.box.map50:.4f} | mAP50-95: {m.box.map:.4f} | precision: {m.box.mp:.4f} | recall: {m.box.mr:.4f}")
per_cls = pd.DataFrame({"class": [CLASSES[i] for i in m.box.ap_class_index],
                        "AP50": m.box.ap50, "AP50-95": m.box.ap}).round(4)
print(per_cls.to_string(index=False))
print("Speed (ms/img):", {k: round(v, 2) for k, v in m.speed.items()})

## 9. Confusion matrix & PR curve (generated by Ultralytics)

In [ ]:
from IPython.display import Image as IPImage, display
for f in ["confusion_matrix_normalized.png", "BoxPR_curve.png"]:
    p = RESULTS_DIR / "test_eval" / f
    if p.exists(): display(IPImage(filename=str(p), width=650))

## 10. Predicted boxes on test boards

In [ ]:
test_imgs = sorted((YOLO_DIR / "images/test").glob("*.*"))
sample = random.sample(test_imgs, min(6, len(test_imgs)))
fig, axes = plt.subplots(2, 3, figsize=(20, 9))
for ax, p in zip(axes.ravel(), sample):
    r = best.predict(str(p), imgsz=IMG_SIZE, conf=0.25, device=device, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(line_width=3, font_size=14), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(p.stem[:40])
plt.tight_layout(); plt.savefig(RESULTS_DIR / "test_predictions.png", dpi=120); plt.show()

## 11. Try your own images (Google boards, new photos)

Out-of-distribution test. Low-resolution boards from a different source will usually score lower than the held-out test boards; report both honestly.

In [ ]:
def detect(path, conf=0.25, imgsz=IMG_SIZE, show=True):
    r = best.predict(str(path), imgsz=imgsz, conf=conf, device=device, verbose=False)[0]
    dets = [{"class": CLASSES[int(b.cls)], "confidence": round(float(b.conf), 4),
             "bbox": [int(v) for v in b.xyxy[0].tolist()]} for b in r.boxes]
    if show:
        plt.figure(figsize=(12, 7)); plt.imshow(cv2.cvtColor(r.plot(line_width=2), cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
    return dets

# print(detect(PROJECT_ROOT / "test_images" / "google_board.jpg", conf=0.15))

## 12. Save detector + metrics summary

In [ ]:
shutil.copy2(best_path, MODEL_DIR / "yolov8_pcb_best.pt")
model_size_mb = os.path.getsize(best_path) / (1024 * 1024)

metrics_summary = {
    "split_id": SPLIT_ID,
    "model": MODEL_NAME.replace(".pt", ""),
    "task": "detection",
    "classes": CLASSES,
    "img_size": IMG_SIZE,
    "map50": round(float(m.box.map50), 4),
    "map50_95": round(float(m.box.map), 4),
    "precision": round(float(m.box.mp), 4),
    "recall": round(float(m.box.mr), 4),
    "per_class_ap50": {CLASSES[i]: round(float(a), 4) for i, a in zip(m.box.ap_class_index, m.box.ap50)},
    "inference_ms": round(float(m.speed["inference"]), 2),
    "model_size_mb": round(model_size_mb, 2),
    "epochs_trained": int(len(res_csv)),
    "training_time_min": round(total_time / 60, 2),
}
with open(RESULTS_DIR / "metrics_summary.json", "w") as f: json.dump(metrics_summary, f, indent=2)
with open(MODEL_DIR / "yolov8_pcb_best.json", "w") as f: json.dump(metrics_summary, f, indent=2)
print(json.dumps(metrics_summary, indent=2))

## 13. Notes / Next steps

- Uses the **same board-level split** as the classifier notebooks (`splits.csv`), so detector and classifier numbers refer to the same unseen boards. With ~10 boards the test mAP is noisy; say so in the report.
- mAP50 is the headline number; mAP50-95 is stricter and will look lower for tiny defects, where a few pixels of box error matter.
- Next: add `POST /api/detect` to the FastAPI backend (full image -> YOLO boxes), then optionally feed each box crop to EfficientNet-B0 as a second opinion on the class.
- Clean-board images (`PCB_USED/`) could be added as background-only training images to cut false positives. Only add the ones from **train** boards, or leakage returns.
- `rotation/` holds rotated copies of the same boards and is not used, for the same leakage reason as in the classifier notebooks.
- Try `IMG_SIZE = 1280` or `yolov8m.pt` if small-defect classes (`mouse_bite`, `spur`, `spurious_copper`) have low AP.